# Topic 5: Optimization Patterns & Production Checklist

> **Phase 4 → Week 7 → Topic 5**
>
> Prerequisites: Topics 1-4 (Shuffle, Skew, AQE, Spark UI)

---

## The Big Picture

You've now learned all the individual tools: shuffle internals, skew fixes, AQE, memory model, executor sizing, GC tuning, spill diagnosis, and Spark UI navigation. This notebook ties everything together into systematic patterns and a repeatable production checklist.

```
Performance Optimization Hierarchy (fix in this order):

  Level 1: Design  ← Biggest impact, hardest to change later
    • Schema design (columnar, appropriate types)
    • Partitioning strategy (partitionBy, bucketing)
    • Join strategy (broadcast small tables)
    • Data format (Parquet > CSV, Delta > Parquet for updates)

  Level 2: Query Logic  ← Medium impact, change at any time
    • Filter early (pushdown predicates)
    • Select only needed columns (projection pruning)
    • Avoid unnecessary shuffles (combine operations)
    • Cache reused DataFrames

  Level 3: Configuration  ← Lower impact, tuning knobs
    • shuffle.partitions (scale with data)
    • executor sizing (cores, memory)
    • AQE enabled
    • GC tuning
```

---

## Interview Cheat Sheet

**Q: Walk me through how you would optimize a slow Spark job.**
> I start with the Spark UI: (1) identify the slowest stage, (2) check for skew (one task >> others), spill (Spill Disk > 0), and too many shuffles (Exchange nodes in SQL tab). Then fix in order of impact: eliminate unnecessary shuffles (broadcast small tables, combine groupBys), fix skew (AQE or salting), fix spill (more partitions), then tune configuration (executor sizing, GC, AQE settings). Code-level fixes come first because they're free; resource increases come last.

**Q: What are common Spark anti-patterns?**
> (1) `collect()` on large DataFrames — brings all data to driver, OOM. Use `write()`. (2) Too many small shuffles — chain multiple groupBys when you can combine. (3) UDFs instead of built-in functions — UDFs are opaque to Catalyst and can't be optimized. (4) `repartition()` unnecessarily — shuffle for no benefit. (5) Not filtering early — shuffling more data than needed. (6) Caching everything — cached data evicts execution memory and causes GC pressure.

**Q: When would you choose coalesce() vs repartition()?**
> `repartition(N)` always does a full shuffle — use when you need to increase or redistribute partitions (e.g., before a join or to fix skew). `coalesce(N)` just merges existing partitions without a shuffle — only decreases partition count, and only use it before writing to output to reduce file count. Never use `coalesce()` if you need data redistribution; it keeps partitions on the same tasks and won't fix skew.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time

spark = SparkSession.builder \
    .appName("Week7 - Optimization Patterns") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Anti-Pattern Hall of Shame

In [ ]:
orders   = spark.read.csv("/workspace/week4/data/orders.csv",   header=True, inferSchema=True)
products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)
customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)

# ─── Anti-Pattern 1: Selecting * then filtering columns later ───────────────
# BAD: reads all columns, then drops them — wastes I/O for column-format files
bad_1 = orders.select("*").filter(F.col("amount") > 100)

# GOOD: select only what you need IMMEDIATELY
good_1 = orders.select("order_id", "customer_id", "amount").filter(F.col("amount") > 100)

print("Anti-Pattern 1: SELECT * when you only need 3 of 7 columns")
print(f"  BAD:  {len(bad_1.columns)} columns in memory")
print(f"  GOOD: {len(good_1.columns)} columns in memory (saves 57% I/O on Parquet)")
print()

In [ ]:
# ─── Anti-Pattern 2: Multiple groupBys that could be one ───────────────────

# BAD: 2 shuffles (2 groupBy operations)
def bad_two_groupby():
    step1 = orders.groupBy("customer_id").agg(F.sum("amount").alias("total"))  # shuffle 1
    step2 = step1.groupBy().agg(F.avg("total").alias("avg_spend"))              # shuffle 2
    return step2

# GOOD: 0 shuffles for the global average (use one aggregation)
def good_one_agg():
    return orders.agg(F.avg("amount").alias("avg_order_amount"))               # no shuffle

print("Anti-Pattern 2: Two groupBys when one agg suffices")
bad_two_groupby().show()
good_one_agg().show()

# Show shuffle count difference
def count_exchanges(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    return plan.count("Exchange")

print(f"  BAD:  {count_exchanges(bad_two_groupby())} shuffles")
print(f"  GOOD: {count_exchanges(good_one_agg())} shuffles")

In [ ]:
# ─── Anti-Pattern 3: Python UDF when a built-in function exists ─────────────
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import time

large_df = spark.range(500_000).withColumn("value", F.rand() * 1000)

# BAD: Python UDF — breaks Catalyst optimization, serializes to Python
@udf(StringType())
def classify_udf(v):
    if v is None: return "Unknown"
    return "High" if v > 750 else "Medium" if v > 250 else "Low"

t0 = time.time()
large_df.withColumn("tier", classify_udf(F.col("value"))).count()
t_udf = time.time() - t0

# GOOD: Spark built-in when/otherwise — stays in JVM, Catalyst-optimizable
t0 = time.time()
large_df.withColumn("tier",
    F.when(F.col("value") > 750, "High")
     .when(F.col("value") > 250, "Medium")
     .otherwise("Low")
).count()
t_builtin = time.time() - t0

print("Anti-Pattern 3: Python UDF vs built-in when/otherwise")
print(f"  Python UDF:     {t_udf:.3f}s")
print(f"  Built-in when:  {t_builtin:.3f}s  ({t_udf/t_builtin:.1f}x faster)")
print()
print("Rule: Always use built-in functions. Only use UDFs when no built-in exists.")
print("      If you must use Python UDFs, prefer Pandas UDFs (Arrow-vectorized).")

In [ ]:
# ─── Anti-Pattern 4: Filter AFTER join instead of BEFORE ──────────────────

# BAD: join all rows, then filter
def bad_filter_after():
    return orders.join(products, on="product_id", how="inner") \
                 .filter(F.col("category") == "Electronics")

# GOOD: filter products first, then join (smaller shuffle)
def good_filter_before():
    electronics = products.filter(F.col("category") == "Electronics")
    return orders.join(electronics, on="product_id", how="inner")

print("Anti-Pattern 4: Filter after join vs filter before join")
print("BAD — join all products first:")
bad_filter_after().explain()

print("GOOD — filter products to Electronics before join:")
good_filter_before().explain()

print()
print("Note: Catalyst WILL push down filters for simple cases automatically.")
print("But explicit filter-before-join is always safer and more readable.")

In [ ]:
# ─── Anti-Pattern 5: repartition before write (creates tiny files) ──────────

print("""
Anti-Pattern 5: Wrong partition count before writing
══════════════════════════════════════════════════════

BAD: Write with 200 shuffle partitions → 200 small files
  orders.write.parquet('/output')  # 200 × 10KB files = Hadoop small file problem

BAD: repartition() before write → unnecessary full shuffle
  orders.repartition(200).write.parquet('/output')  # shuffle for no reason

GOOD: coalesce() before write → no shuffle, fewer files
  orders.coalesce(4).write.parquet('/output')  # 4 files, no shuffle

GOOD: partitionBy() for future query pushdown
  orders.write.partitionBy('status').parquet('/output')
  # Creates /output/status=delivered/, /output/status=cancelled/
  # Future filter(status='delivered') skips other directories

BEST: AQE coalesces automatically during job
  spark.conf.set('spark.sql.adaptive.enabled', 'true')
  # AQE reduces shuffle partitions automatically — often no need for manual coalesce
══════════════════════════════════════════════════════
""")

## Part 2: Positive Patterns — Do These

In [ ]:
# Pattern 1: Broadcast small dimension tables
print("Pattern 1: Broadcast small tables in joins")

# Products table is small (10 rows) — always broadcast it
q_with_broadcast = orders \
    .join(broadcast(products), on="product_id", how="inner") \
    .join(broadcast(customers), on="customer_id", how="left")

print(f"With broadcast: {count_exchanges(q_with_broadcast)} shuffles (only orders shuffles)")

q_no_broadcast = orders \
    .join(products, on="product_id", how="inner") \
    .join(customers, on="customer_id", how="left")

print(f"No broadcast:   {count_exchanges(q_no_broadcast)} shuffles")
print()
print("Rule: tables < 10MB = always broadcast. 10-100MB = consider broadcast.")

In [ ]:
# Pattern 2: Cache strategically — reused intermediate results
print("Pattern 2: Cache reused intermediate DataFrames")

from pyspark import StorageLevel

# Expensive join used 3 times
enriched_orders = orders \
    .join(broadcast(products), on="product_id") \
    .join(broadcast(customers), on="customer_id") \
    .cache()  # cache BEFORE using 3 times

enriched_orders.count()  # trigger caching

t0 = time.time()
q1 = enriched_orders.groupBy("category").agg(F.sum("amount"))
q2 = enriched_orders.groupBy("tier").agg(F.avg("amount"))
q3 = enriched_orders.groupBy("status").count()

q1.show(5)
q2.show(5)
q3.show(5)
print(f"3 queries on cached data: {time.time()-t0:.3f}s")

enriched_orders.unpersist()
print("Unpersisted after use — always clean up!")

In [ ]:
# Pattern 3: Use explain() proactively during development
print("Pattern 3: Always check explain() before running large queries")

query = orders \
    .filter(F.col("amount") > 100) \
    .join(broadcast(products.select("product_id", "category")), on="product_id") \
    .groupBy("category") \
    .agg(F.sum("amount").alias("total"), F.count("*").alias("cnt")) \
    .orderBy(F.col("total").desc())

print("Exchanges in plan:", count_exchanges(query))
query.explain()

print("""
What to verify in explain() before running:
  ✓ Filter appears BEFORE Exchange (pushdown working)
  ✓ Small tables use BroadcastHashJoin, not SortMergeJoin
  ✓ Number of Exchange nodes matches expectation
  ✓ Column projection applied (only needed columns appear)
  ✓ No CartesianProduct (accidental cross join)
""")

In [ ]:
# Pattern 4: Set job descriptions for UI navigation
print("Pattern 4: Name your jobs for easy Spark UI navigation")

# Always set a description before important jobs
spark.sparkContext.setJobDescription("ETL Step 1: Load and validate orders")
order_count = orders.count()

spark.sparkContext.setJobDescription("ETL Step 2: Enrich with product categories")
enriched = orders.join(broadcast(products), on="product_id")
enriched.count()

spark.sparkContext.setJobDescription("ETL Step 3: Aggregate by category and write")
result = enriched.groupBy("category").agg(F.sum("amount").alias("total"))
result.show()

spark.sparkContext.setJobDescription("")  # reset

print()
print("In Spark UI → Jobs tab, each job now has a readable name.")
print("Makes it trivial to find which step is slow.")

## Part 3: Complete Production Checklist

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           SPARK PRODUCTION OPTIMIZATION CHECKLIST                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  CONFIGURATION (set once, applies to all jobs)                       ║
║  ─────────────────────────────────────────────                       ║
║  □ spark.sql.adaptive.enabled = true                                 ║
║  □ spark.sql.adaptive.coalescePartitions.enabled = true              ║
║  □ spark.sql.adaptive.skewJoin.enabled = true                        ║
║  □ spark.sql.shuffle.partitions = 2-4× total executor cores          ║
║  □ spark.executor.extraJavaOptions = -XX:+UseG1GC                    ║
║    -XX:InitiatingHeapOccupancyPercent=35                             ║
║  □ Executor sizing: 4-5 cores, 8-16GB each                           ║
║  □ spark.sql.autoBroadcastJoinThreshold = 10MB (adjust to cluster)   ║
║                                                                      ║
║  BEFORE WRITING A QUERY                                              ║
║  ─────────────────────────────                                       ║
║  □ Run groupBy on join keys — check skew ratio (alert: > 10×)        ║
║  □ Check table sizes — can any be broadcast?                         ║
║  □ Identify all wide transformations (shuffles) in your logic        ║
║  □ Plan your filter order: filter early, reduce data before shuffle  ║
║                                                                      ║
║  WHILE WRITING CODE                                                  ║
║  ─────────────────                                                   ║
║  □ Select only needed columns immediately after read                 ║
║  □ Filter before join/groupBy where possible                         ║
║  □ Use broadcast() on small tables (< 10MB)                          ║
║  □ Use built-in functions over Python UDFs                           ║
║  □ Combine multiple groupBys on the same key into one                ║
║  □ Cache intermediate results used 2+ times; unpersist when done     ║
║  □ Set setJobDescription() for each logical step                     ║
║  □ Call explain() before running large queries                       ║
║                                                                      ║
║  AFTER RUNNING                                                       ║
║  ─────────────                                                       ║
║  □ Check Spark UI: any stage > 10min?                                ║
║  □ Check Stages tab: Spill (Disk) > 0 anywhere?                      ║
║  □ Check Stage Detail: one task >> others (skew)?                    ║
║  □ Check Executors tab: any dead executors? GC > 10%?                ║
║  □ Count Exchange nodes in SQL tab: can you reduce them?             ║
║                                                                      ║
║  WRITING OUTPUT                                                      ║
║  ──────────────                                                      ║
║  □ Use Parquet or Delta (not CSV/JSON) for large data                ║
║  □ partitionBy() logical columns for future query pushdown           ║
║  □ coalesce() before write to control output file count              ║
║  □ NEVER collect() large DataFrames — use write() instead            ║
╚══════════════════════════════════════════════════════════════════════╝
""")

## Part 4: Interview Q&A Marathon

In [ ]:
print("""
Phase 4 Interview Questions — Complete Coverage:
═══════════════════════════════════════════════════════════════════

SHUFFLE (Week 7 Topic 1)
Q: What triggers a shuffle in Spark?
A: Wide transformations that require co-locating rows with the same key:
   groupBy, agg, join (SMJ), distinct, repartition, orderBy, coalesce(sometimes),
   intersection, subtract. Narrow transformations (map, filter, union) do not.

Q: How many shuffle files does Sort-Merge shuffle create?
A: M × R — M mapper tasks × R reduce partitions. 100 mappers × 200 partitions
   = 20,000 files. Each file is an indexed + data file pair.

SKEW (Week 7 Topic 2)
Q: How do you fix data skew in a join?
A: (1) Enable AQE skew join (free, handles most cases). (2) Broadcast the small
   side (eliminates skew by avoiding shuffle entirely). (3) Salt the hot key:
   add random suffix 0..N to left side, replicate right side N times, join on
   salted key, then re-aggregate to strip the salt.

AQE (Week 7 Topic 3)
Q: What are the three AQE features?
A: (1) Coalescing shuffle partitions — merges tiny post-shuffle partitions
   (2) Dynamic join strategy — switches SMJ→BHJ when one side turns out small
   (3) Skew join — splits skewed partitions into sub-tasks automatically

SPARK UI (Week 7 Topic 4)
Q: Where in Spark UI do you look to diagnose a slow job?
A: Jobs → find slow job → Stages → find slow stage → Stage Detail → Tasks.
   Check: Duration outlier (skew), Spill Disk > 0 (memory), GC% (GC pressure).
   Also: SQL tab for Exchange count and join strategies.

MEMORY MODEL (Week 6 Topic 1)
Q: What happens when execution memory needs more space than available?
A: It evicts LRU cached partitions from storage memory. If still not enough,
   it spills to disk. If even that's not possible (e.g., broadcast), OOM.

EXECUTOR SIZING (Week 6 Topic 2)
Q: How many cores per executor is optimal?
A: 4-5 cores. Too few = parallelism is underutilized per executor. Too many
   (> 5) = HDFS throughput bottleneck and excessive GC on large heaps.

GC (Week 6 Topic 3)
Q: What GC is recommended for Spark and why?
A: G1GC (-XX:+UseG1GC). It handles large heaps better than CMS/Parallel GC
   and produces shorter, more predictable pauses. Combined with
   InitiatingHeapOccupancyPercent=35 to start GC earlier and avoid long pauses.

SPILL & OOM (Week 6 Topic 4)
Q: What's the 4-step OOM diagnosis workflow?
A: (1) Locate OOM: driver vs executor? (2) Driver path: check collect() or
   large broadcast. (3) Executor path: check skew, broadcast size, spill.
   (4) Fix in order: more partitions → fix skew → remove broadcast → more memory.

BROADCAST (Week 6 Topic 5)
Q: Difference between sc.broadcast() and F.broadcast()?
A: sc.broadcast(obj) creates a broadcast variable (Python dict/object) sent
   once per executor, accessed via .value in RDD ops/UDFs. F.broadcast(df)
   is a Catalyst hint for BroadcastHashJoin during DataFrame join execution.
   Different mechanisms, different use cases.
═══════════════════════════════════════════════════════════════════
""")

## Part 5: Putting It All Together — Full ETL Example

In [ ]:
# A well-optimized ETL pipeline applying all patterns

spark.sparkContext.setJobDescription("Optimized ETL: Revenue by region and category")

# ── STEP 1: Load with projection (only needed columns) ──────────────────────
orders_slim = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True) \
    .select("order_id", "customer_id", "product_id", "amount", "status") \
    .filter(F.col("status").isin("delivered", "pending"))  # filter early

# Small dimension tables — broadcast both
products_slim = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True) \
    .select("product_id", "category")

customers_slim = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True) \
    .select("customer_id", "country")

# ── STEP 2: Check skew on join keys before joining ───────────────────────────
print("Pre-join skew check on customer_id:")
orders_slim.groupBy("customer_id").count() \
           .agg(F.max("count").alias("max"),
                F.avg("count").alias("avg"),
                F.min("count").alias("min")).show()

# ── STEP 3: Broadcast joins (no shuffle for dimension tables) ─────────────────
enriched = orders_slim \
    .join(broadcast(products_slim), on="product_id", how="left") \
    .join(broadcast(customers_slim), on="customer_id", how="left")

# ── STEP 4: Cache if used multiple times ─────────────────────────────────────
enriched.cache()
enriched.count()  # materialize cache

# ── STEP 5: Multiple aggregations on cached result ───────────────────────────
print("\nRevenue by category:")
enriched.groupBy("category").agg(
    F.sum("amount").alias("revenue"),
    F.count("*").alias("orders")
).orderBy(F.col("revenue").desc()).show()

print("Revenue by country:")
enriched.groupBy("country").agg(
    F.sum("amount").alias("revenue")
).orderBy(F.col("revenue").desc()).show()

# ── STEP 6: Clean up cache ───────────────────────────────────────────────────
enriched.unpersist()

# ── STEP 7: Write result (coalesce to control file count) ────────────────────
final_result = enriched \
    .groupBy("category", "country") \
    .agg(F.sum("amount").alias("revenue"))

final_result.coalesce(2).write.mode("overwrite").parquet("/tmp/revenue_by_category_country")
print("\nWritten to /tmp/revenue_by_category_country (2 files)")
print()
print("Patterns applied:")
print("  ✓ Early projection (select only needed columns)")
print("  ✓ Early filter (only delivered/pending orders)")
print("  ✓ Broadcast joins for small dimension tables")
print("  ✓ Pre-join skew check")
print("  ✓ Cache for multi-use intermediate result")
print("  ✓ unpersist after done")
print("  ✓ coalesce before write")
print("  ✓ Parquet output format")

## Exercises

1. Take the following query and apply at least 4 optimizations from the checklist. Count how many shuffles it had before and after.
   ```python
   result = spark.read.csv('/workspace/week4/data/orders.csv', header=True, inferSchema=True) \
       .join(spark.read.csv('/workspace/week4/data/products.csv', header=True, inferSchema=True), on='product_id') \
       .join(spark.read.csv('/workspace/week4/data/customers.csv', header=True, inferSchema=True), on='customer_id') \
       .filter(F.col('category') == 'Electronics') \
       .groupBy('country').agg(F.sum('amount'))
   ```
2. Write a function `profile_query(df, label)` that: (a) calls `explain()` and counts Exchange nodes, (b) times the execution, (c) prints a summary. Use it to compare 3 versions of the same query.
3. When should you NOT use `broadcast()`? Give 3 specific situations.
4. You have a 1TB dataset partitioned on disk by `country`. Write the optimal Spark code to compute total revenue per product category for only USA orders.
5. A colleague says "I always set `shuffle.partitions=1000` to be safe." What's wrong with this and how would you explain the right approach?

In [ ]:
# Exercise 2: query profiler utility
def profile_query(df, label):
    """Profile a DataFrame query: shuffle count + timing."""
    plan = df._jdf.queryExecution().executedPlan().toString()
    exchanges = plan.count("Exchange")
    t0 = time.time()
    count = df.count()
    elapsed = time.time() - t0
    print(f"[{label}]")
    print(f"  Shuffles (Exchange nodes): {exchanges}")
    print(f"  Row count: {count:,}")
    print(f"  Time: {elapsed:.3f}s")
    print()

# Compare 3 versions
orders_all = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
products_all = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)

# Version A: no optimization
v_a = orders_all.join(products_all, on="product_id") \
                .filter(F.col("category") == "Electronics")
profile_query(v_a, "A: No optimization (filter after join)")

# Version B: filter before join
v_b = orders_all.join(
    products_all.filter(F.col("category") == "Electronics"),
    on="product_id"
)
profile_query(v_b, "B: Filter before join")

# Version C: filter before + broadcast
v_c = orders_all.join(
    broadcast(products_all.filter(F.col("category") == "Electronics")),
    on="product_id"
)
profile_query(v_c, "C: Filter before + broadcast (optimal)")

In [ ]:
# Exercise 3: When NOT to broadcast
print("""
When NOT to use broadcast():

1. Table is large (> executor memory / 2)
   broadcast() forces the data into executor heap — no spill path.
   If the table is 8GB and executor has 8GB heap, OOM is guaranteed.
   Rule: only broadcast if table comfortably fits in executor memory.

2. Both sides are large (SortMergeJoin is more memory-efficient)
   If both sides are 50GB, you can't broadcast either. SMJ uses disk-backed
   sort buffers that can spill; BHJ cannot spill.

3. Dynamic allocation + aggressive executor release
   Broadcast data is held per executor. When executors are released
   (dynamic allocation), the data is lost. Future tasks on new executors
   must re-broadcast. This can cause excessive re-broadcast overhead.
   (Less of a problem with Spark 3.2+ broadcast caching improvements.)
""")

# Exercise 5: Why shuffle.partitions=1000 is wrong
print("""
Why shuffle.partitions=1000 is problematic:

With 1000 partitions and 50MB of shuffle data:
  Average partition = 50MB / 1000 = 50KB
  Task scheduling overhead for 1000 tasks >> actual computation time
  1000 tasks × (scheduler delay + deserialization + result return)
  = overhead dominates

With AQE enabled (the right approach):
  Set shuffle.partitions=200 (or 2-4× executor cores)
  AQE will coalesce to the right number based on actual data size
  → No more wasted tiny tasks
  → No need to manually tune per-job partition count

The correct conversation:
  - Set shuffle.partitions = 2-4× executor cores (your baseline)
  - Enable AQE coalescing (auto-reduces for small data)
  - For known large jobs: increase shuffle.partitions explicitly
  - AQE will never INCREASE partition count — so set your baseline high enough
""")